# Improving AM - Grid Search (Colab)

This notebook runs a deterministic grid search for Activation Maximization (AM) using the project's AM engine.
It mounts Google Drive, installs dependencies if needed, runs the grid script, and copies results to Drive.



In [ ]:
#@title Setup: Mount Drive and install dependencies
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

!pip -q install pyyaml h5py torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu

print('✅ Drive mounted and dependencies installed')


In [ ]:
REPO_URL = 'https://github.com/assafzimand/Physics-Informed-DL-Project.git'
BRANCH = 'main'
CLONE_DIR = '/content/project'

if os.path.exists(CLONE_DIR):
  %cd $CLONE_DIR
  !git fetch origin $BRANCH
  !git checkout $BRANCH
  !git reset --hard origin/$BRANCH
else:
  !git clone --depth=1 --branch $BRANCH $REPO_URL $CLONE_DIR
  %cd $CLONE_DIR

print('📁 CWD:', os.getcwd())

In [ ]:
#@title Run grid (baseline) and save to Drive
# Provide explicit model path (e.g., from your Drive)
MODEL_PATH = '/content/drive/MyDrive/models/cv_full_5fold_75epochs_fold_2_best.pth'  #@param {type:"string"}

!python scripts/improving_am/run_grid.py \
  --config configs/improving_am/baseline.yaml \
  --out_dir experiments/improving_am \
  --seed 42 \
  --model_path "$MODEL_PATH" | cat

# After run completes, archive and copy to Drive automatically
import os, shutil
from pathlib import Path

base = Path('experiments/improving_am')
if base.exists():
  latest = sorted(base.glob('*'), key=os.path.getmtime)[-1]
  zip_path = Path('/content/improving_am_results.zip')
  shutil.make_archive(str(zip_path.with_suffix('')), 'zip', latest)
  DRIVE_OUT_DIR = Path('/content/drive/MyDrive/AM_Results')
  DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
  shutil.copy2(zip_path, DRIVE_OUT_DIR / zip_path.name)
  print('✅ Saved results to Drive:', DRIVE_OUT_DIR / zip_path.name)
else:
  print('⚠️ No results directory found; nothing to save')
